In [2]:
import pandas as pd

In [28]:
df_ctrl = pd.read_excel("outputs/control_clustered.xlsx")

In [29]:
df_ctrl.head()

,Unnamed: 0,text,kmeans_cluster_id,hdbscan_cluster_id
0,0,Coordination of simultaneous operations (SIMOP...,253,-1
1,1,Conducting of risk assessments (TRA / JSA / LMRA),175,44
2,2,Provision of spill kits / spill containment,26,43
3,3,Carrying out of pre-use / pre-operational equi...,15,14
4,4,Wearing of fall protection / harness,308,26


In [30]:
df_ctrl[['kmeans_cluster_id', 'hdbscan_cluster_id']] = df_ctrl[['kmeans_cluster_id', 'hdbscan_cluster_id']].astype('int64')

In [31]:
nr_of_controls = df_ctrl['hdbscan_cluster_id'].count()
print(f"Total number of controls: {nr_of_controls}")

Total number of controls: 1199


In [32]:
print(f"HDBSCAN results in {len(df_ctrl['hdbscan_cluster_id'].value_counts()) - 1} clusters.")
print(f"KMeans  results in {len(df_ctrl['kmeans_cluster_id'].value_counts())} clusters.")


HDBSCAN results in 51 clusters.
KMeans  results in 400 clusters.


397

In [40]:
hdbscan_unclustered = df_ctrl[df_ctrl['hdbscan_cluster_id'] == -1]
print(f"HDBSCAN clustered {((nr_of_controls - hdbscan_unclustered.shape[0]) / nr_of_controls) * 100:.2f}% of all controls ({hdbscan_unclustered.shape[0]} of {nr_of_controls} left unclustered).")

HDBSCAN clustered 66.89% of all controls (397 of 1199 left unclustered).


In [42]:
# Create a third clustering, based on the overlap between the two clusters.

# Set default value to -1 (i.e. not clusterable)
df_ctrl['diff_cluster'] = -1

next_id = 0
for hdb_id, group in df_ctrl[df_ctrl['hdbscan_cluster_id'] != -1].groupby('hdbscan_cluster_id'):
    for kmeans_id, subgroup in group.groupby('kmeans_cluster_id'):
        if len(subgroup) >= 2:
            df_ctrl.loc[subgroup.index, 'diff_cluster'] = next_id
            next_id += 1

df_ctrl['diff_cluster'] = df_ctrl['diff_cluster'].astype('int64')
df_ctrl.head()

,Unnamed: 0,text,kmeans_cluster_id,hdbscan_cluster_id,diff_cluster
0,0,Coordination of simultaneous operations (SIMOP...,253,-1,-1
1,1,Conducting of risk assessments (TRA / JSA / LMRA),175,44,-1
2,2,Provision of spill kits / spill containment,26,43,148
3,3,Carrying out of pre-use / pre-operational equi...,15,14,45
4,4,Wearing of fall protection / harness,308,26,94


In [45]:
diff_unclustered = df_ctrl[df_ctrl['diff_cluster'] == -1]
print(f"Agreement clustered {((nr_of_controls - diff_unclustered.shape[0]) / nr_of_controls) * 100:.2f}% of all controls ({diff_unclustered.shape[0]} of {nr_of_controls} left unclustered).")

Agreement clustered 53.96% of all controls (552 of 1199 left unclustered).


In [46]:
df_ctrl.to_excel("outputs/agreement_cluster.xlsx", index=False)